# tSCS EMG — the four comparison figures (subject NTA, 24-07-2026)

One session, one participant: **stimulation mode** (30 Hz burst / ARC-EX) × **polarity**
(cathodic / anodic) × **lidocaine** (before / with). Four figures, each holding one thing fixed:

| figure | fixed | compared | with |
|---|---|---|---|
| **1** | anodic | 30 Hz vs ARC-EX | before and with lidocaine |
| **2** | cathodic | 30 Hz vs ARC-EX | before and with lidocaine |
| **3** | ARC-EX | anodic vs cathodic | before and with lidocaine |
| **4** | 30 Hz burst | anodic vs cathodic | before and with lidocaine |

## Colours and styles
**gray = before lidocaine, orange = with lidocaine** in every figure. The second factor is the
**style**: in figures 1–2 **solid / plain bars = 30 Hz burst, dashed / hatched = ARC-EX**; in
figures 3–4 **solid / plain = cathodic, dashed / hatched = anodic**.

**Intensities:** `BURST_MA` and `ARCEX_MA` in the config set the mA of 30 Hz and of ARC-EX in
every figure; the `AMP[...]` lines below them override any single train (polarity × lidocaine)
when you want each at its own motor threshold — the log gives burst cathodic 25 mA before / 30
with lidocaine, anodic 30 / 30; ARC-EX cathodic 70 / 70, anodic 65 / 90. `FIGS[n]` sets the muscle
of each figure's one-muscle version (and can override its intensities too).

Every figure also comes with **waterfalls** — all intensities of each condition stacked, and the
four overlaid — so you can see the whole sweep before choosing the mA to compare at.

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from functions import set_style, load_run, pretty, waterfall, waterfall_overlay, detect_pulses
from functions.burst import (compare_at_intensity, summary_curves, plot_pulse_overlay,
                             burst_p2p, detection_report, resolve_muscles, artifact_extent)
from functions.paper import (fig_train_modes, fig_depression, depression_stats,
                             motor_thresholds, muscles_with_threshold, fig_thresholds)
set_style()


## Config

In [ ]:
D = "tSCS_CHUV_data/24-07-2026/testSCS/"
FILE = {   # (mode, polarity, lidocaine state) -> file
    ("burst", "cathodic", "before"):    "Burst_autosave_20260724_102443_670ms.csv",
    ("burst", "cathodic", "lidocaine"): "Burst_autosave_20260724_113834_522ms.csv",
    ("burst", "anodic",   "before"):    "Burst_autosave_20260724_102721_694ms.csv",
    ("burst", "anodic",   "lidocaine"): "Burst_autosave_20260724_114055_891ms.csv",
    ("arcex", "cathodic", "before"):    "Modulated_autosave_20260724_103530_508ms.csv",
    ("arcex", "cathodic", "lidocaine"): "Modulated_autosave_20260724_114332_473ms.csv",
    ("arcex", "anodic",   "before"):    "Modulated_autosave_20260724_103849_459ms.csv",
    ("arcex", "anodic",   "lidocaine"): "Modulated_autosave_20260724_114730_561ms.csv",
}
NAME = {"burst": "30 Hz", "arcex": "ARC-EX", "cathodic": "cathodic", "anodic": "anodic"}
LIDO_COL = {"before": "0.45", "lidocaine": "#f39c12"}       # colour = lidocaine
STYLE_A  = ("",    "-")                                      # first compared value: plain / solid
STYLE_B  = ("///", "--")                                     # second: hatched / dashed

# ---- the intensity of every train, in one table -------------------------------------------
BURST_MA, ARCEX_MA = 35, 110      # <-- the mA used for 30 Hz and for ARC-EX, everywhere
AMP = {k: (BURST_MA if k[0] == "burst" else ARCEX_MA) for k in FILE}

# per-condition overrides - uncomment/edit any line to give that train its own mA
AMP[("burst", "cathodic", "before")]    = 35      # log motor threshold: 25
AMP[("burst", "cathodic", "lidocaine")] = 35
AMP[("burst", "anodic",   "before")]    = 35
AMP[("burst", "anodic",   "lidocaine")] = 35
AMP[("arcex", "cathodic", "before")]    = 75
AMP[("arcex", "cathodic", "lidocaine")] = 75
AMP[("arcex", "anodic",   "before")]    = 105
AMP[("arcex", "anodic",   "lidocaine")] = 105

FIG_SETTINGS = {   # <-- everything each figure plots: the mA of each compared condition,
                   #     the muscle of its one-muscle version, and its waterfall muscles
    1: dict(amps={"burst": BURST_MA, "arcex": ARCEX_MA},
            muscle="Flex. digitorum (R)", wf=None),   # fig 1: anodic · 30 Hz vs ARC-EX
    2: dict(amps={"burst": BURST_MA, "arcex": ARCEX_MA},
            muscle="Flex. digitorum (R)", wf=None),   # fig 2: cathodic · 30 Hz vs ARC-EX
    3: dict(amps={"cathodic": 105, "anodic": 80},
            muscle="Flex. digitorum (R)", wf=None),   # fig 3: ARC-EX · cathodic vs anodic
    4: dict(amps={"cathodic": BURST_MA, "anodic": BURST_MA},
            muscle="Flex. digitorum (R)", wf=None),   # fig 4: 30 Hz burst · cathodic vs anodic
}

FIGS = {   # what each figure holds fixed and what it compares (fixed by the design, not settings)
    1: dict(fixed=("polarity", "anodic"),   compare=("burst", "arcex")),
    2: dict(fixed=("polarity", "cathodic"), compare=("burst", "arcex")),
    3: dict(fixed=("mode", "arcex"),        compare=("cathodic", "anodic")),
    4: dict(fixed=("mode", "burst"),        compare=("cathodic", "anodic")),
}
XLIM_WF   = (-20, 130)      # time window of the waterfalls (ms)
WF_EACH   = True            # also draw each condition's waterfall on its own, not only the overlay
WF_MUSCLES = None           # None = all muscles in the waterfalls, or a list, e.g. ["Flex. digitorum (R)"]

# ---- peak-detection parameters (see the section "How the peaks are detected" below) --------
N_PULSES      = 10      # pulses of the train analysed
RESP_START_MS = 8.0     # response window starts this long after EACH pulse onset - must clear the
                        # artifact. A dict overrides single channels: {"R_DELmed": 11.0}
RESP_END_MS   = None    # window end, also from the pulse onset; None = run to the next pulse
GUARD_MS      = 1.0     # ...minus this
SNR_ON        = "median" # which pulses the criterion looks at: "median" (the train's median
                        # pulse), "p1" (pulse 1 only), "half" (at least half the pulses).
                        # "p1" throws away a whole train when its first pulse happens to be small.
MIN_SNR       = 1.2     # keep a train only if pulse-1 p2p >= MIN_SNR x baseline p2p (None = keep all)
ANCHOR, ANCHOR_WIN_MS = "mean", 3.0   # anchor each pulse's max/min to the train average
MAX_EDGE_FRAC = 0.5     # reject a train as artifact if more than this fraction of its pulses have
                        # max/min on a window border (None = off)
EDGE_MS       = 1.0     # "on a border" means within this many ms of it
JITTER_MS     = 0.5     # flag a pulse whose peak latency differs from the train median by more than this
KW = dict(n_pulses=N_PULSES, resp_start_ms=RESP_START_MS, resp_end_ms=RESP_END_MS,
          guard_ms=GUARD_MS, min_snr=MIN_SNR, max_edge_frac=MAX_EDGE_FRAC, snr_on=SNR_ON,
          anchor=ANCHOR, anchor_win_ms=ANCHOR_WIN_MS)

muscles = [c for c in load_run(D + FILE[("burst", "cathodic", "before")])[2] if c != "Trigger A"]

def build(n, amps=None, muscle=None):
    """(amps / muscle default to FIG_SETTINGS[n] above.)"""
    """Everything figure `n` needs, ordered A-before, A-lidocaine, B-before, B-lidocaine.
    amps: {compared value: mA} for this figure (default: the AMP table above).
    muscle: the muscle of the one-muscle version and of the focused waterfalls."""
    spec = FIGS[n]; fixed, compare = spec["fixed"], spec["compare"]
    amps = amps if amps is not None else FIG_SETTINGS[n]["amps"]
    use_mt = amps == "MT"          # each muscle at its own motor threshold
    muscle = muscle if muscle is not None else FIG_SETTINGS[n]["muscle"]
    files, labels, colours, hatches, lss, amp_list, keys = [], [], [], [], [], [], []
    for val, (hat, ls) in zip(compare, (STYLE_A, STYLE_B)):
        for state in ("before", "lidocaine"):
            key = (val, fixed[1], state) if fixed[0] == "polarity" else (fixed[1], val, state)
            keys.append(key); files.append(D + FILE[key])
            labels.append(f"{NAME[key[0]]} · {key[1]} · {'before lido' if state == 'before' else 'with lido'}")
            colours.append(LIDO_COL[state]); hatches.append(hat); lss.append(ls)
            amp_list.append(MT[key] if use_mt else amps.get(val, AMP[key]))
    return dict(files=files, labels=labels, colours=colours, hatches=hatches, linestyles=lss,
                amps=tuple(amp_list), keys=keys, muscle=muscle, n=n)

def peaks(cfg, muscle=None, report=True, kw=None):
    """Look at HOW the peaks are detected for this figure, at the mA it uses.
    One pulse-overlay per condition: every pulse re-aligned to its own onset, green = the response
    window, v / ^ = the max / min taken, red ring = flagged (on a window border, or at a different
    latency than the other pulses). Stacked markers = the same deflection every time.
    muscle: one label / list (default: the figure's muscle) · kw: override the detection parameters."""
    kw = kw or KW
    ms = muscle or cfg["muscle"] or muscles
    for key, f_, a in zip(cfg["keys"], cfg["files"], cfg["amps"]):
        meta_, t_, sig_ = load_run(f_)
        lab = f"{NAME[key[0]]} · {key[1]} · {'before lido' if key[2] == 'before' else 'with lido'}"
        plot_pulse_overlay(meta_, t_, sig_, ms, amp=a, edge_ms=EDGE_MS, jitter_ms=JITTER_MS,
                           title=f"{lab} — {a} mA", **kw)
        if report:
            chans = resolve_muscles([c for c in sig_ if c != "Trigger A"], ms if isinstance(ms, list) else [ms])
            res = burst_p2p(meta_, t_, sig_, chans, **kw)
            print(f"----- {lab} — {a} mA")
            detection_report(res, chans, edge_ms=EDGE_MS, jitter_ms=JITTER_MS)
            print(f"  artifact spike width (ms after onset): "
                  + ", ".join(f"{pretty(m)} {v:.1f}" for m, v in
                              artifact_extent(t_, sig_, chans, detect_pulses(t_, sig_["Trigger A"])[:N_PULSES],
                                              w=[x["amp_ma"] for x in meta_].index(a)).items()))

def check(cfg):
    """Print what each train of this figure will be plotted at, and whether that mA exists."""
    for key, f_, a in zip(cfg["keys"], cfg["files"], cfg["amps"]):
        avail = [m["amp_ma"] for m in load_run(f_)[0]]
        print(f"{NAME[key[0]]:7s} {key[1]:9s} {key[2]:10s} -> {a:4d} mA "
              + ("ok" if a in avail else f"MISSING - available: {avail}"))

def show(cfg, title, muscle=None):
    """The comparison figure; muscle=cfg["muscle"] for the one-muscle version."""
    compare_at_intensity(cfg["files"], None, amp=cfg["amps"], normalize="none", muscles=muscle,
                         labels=cfg["labels"], colours=cfg["colours"], hatches=cfg["hatches"],
                         linestyles=cfg["linestyles"], title=title, **KW)

def wf(cfg, prefix, muscles_=None, each=None):
    """Waterfalls: every intensity of each condition, titled with mode · polarity · lidocaine,
    then all four overlaid. One shared gain per muscle, so amplitudes are comparable."""
    ms = muscles_ or WF_MUSCLES or muscles
    runs = [load_run(f) for f in cfg["files"]]
    gains = waterfall(*runs[0], ms, xlim=XLIM_WF,
                      title=f"{prefix} · waterfall — {cfg['labels'][0]}  (sets the gain)")
    if WF_EACH if each is None else each:
        for lab, r in zip(cfg["labels"][1:], runs[1:]):
            waterfall(*r, ms, xlim=XLIM_WF, gains=gains, title=f"{prefix} · waterfall — {lab}")
    waterfall_overlay(runs, muscles=ms, xlim=XLIM_WF, gains=gains, labels=cfg["labels"],
                      colours=cfg["colours"], linestyles=cfg["linestyles"],
                      title=f"{prefix} · waterfalls overlaid — all four conditions")

for k, f_ in FILE.items():                        # what each recording contains
    print(f"{NAME[k[0]]:7s} {k[1]:9s} {k[2]:10s} {[m['amp_ma'] for m in load_run(D + f_)[0]]} mA")


## How the peaks are detected — and how to check it

For every pulse the peak-to-peak is `max − min` inside a window that starts **after that pulse's
artifact**:

    window_k = [ pulse_k onset + RESP_START_MS , pulse_(k+1) onset − GUARD_MS ]   (or + RESP_END_MS)

Then two filters: a train counts as a response only if **pulse-1 p2p ≥ `MIN_SNR` × the
pre-stimulus baseline p2p**, and a train is **rejected as artifact** if more than `MAX_EDGE_FRAC`
of its pulses have their max/min sitting within `EDGE_MS` of a window border (a smooth
artifact-recovery curve has no peak inside the window, so its extremes fall on the edges).

| parameter | raise it when | lower it when |
|---|---|---|
| `RESP_START_MS` | the ▼/▲ land on the artifact decay right after the pulse | the window cuts off the start of a real response |
| `RESP_END_MS` | — | the window catches something late that isn't the response |
| `MIN_SNR` | noise-only trains are being kept | real small responses are dropped |
| `MAX_EDGE_FRAC` | a real response is being called artifact | artifact recovery is being kept |
| `ANCHOR` | the detector jumps between two candidate peaks | you want a free search |
| `JITTER_MS` | too many pulses flagged on a variable-latency muscle | you want stricter flagging |

`peaks(FIGn)` below draws, for each condition **at the mA that figure uses**, every pulse
re-aligned to its own onset: **green = the response window, ▼/▲ = the max/min actually taken, red
ring = flagged**. If the detection is right, the markers stack on top of each other. It also
prints the flag counts and the measured artifact width per channel, so you can set
`RESP_START_MS` above it.

To try other settings without touching the config, pass `kw=`:
```python
peaks(FIG1, kw=dict(KW, resp_start_ms=11.0, min_snr=3.0))
```

**Which max and min each pulse gets (`ANCHOR`)** — when a waveform has two candidate peaks (say a
dip at 11 ms and another at 20 ms around a peak at 15 ms), a plain `argmax`/`argmin` picks
whichever happens to be larger on that pulse, so it jumps between them and the pulses get flagged
for "jitter" even though the response is perfectly consistent. With `ANCHOR`, the latency of the
max and of the min is taken **from the train itself** — the average of its pulses (`"mean"`), or
the first one / first two (`"first"`, `"first2"`) — and each pulse is then searched only within
`ANCHOR_WIN_MS` of those latencies. Every pulse is measured on the *same* deflection.
`ANCHOR = None` restores the independent per-pulse search.

**A floor on `JITTER_MS`.** Latencies can only differ in whole samples (here **0.79 ms**), so a
tolerance below that flags a pulse that is one sample off the median — which is why 0.5 ms flags
almost everything. And with `ANCHOR` on, every pulse is already searched within `ANCHOR_WIN_MS` of
the same reference, so a tighter tolerance cannot mean anything. The flag rule therefore applies
`max(JITTER_MS, 1.5 × sampling interval, ANCHOR_WIN_MS)` and prints the value it used.

**Which pulses the criterion looks at (`SNR_ON`).** Testing **pulse 1 only** rejects a whole train
whenever its first pulse happens to be small — e.g. ARC-EX cathodic 110 mA on Flex. digitorum (R),
where pulse 1 sits at SNR 1.16 while the other nine are at 1.9–3.8 and the response is obvious.
`SNR_ON = "median"` (default) uses the train's median pulse, `"half"` asks that at least half the
pulses clear the threshold, `"p1"` is the old behaviour. When a train is kept but its **first**
pulse is near noise, the diagnostics print a warning — the *% of pulse 1* numbers are unreliable
for that train, even though its raw mV values are fine.


## Motor threshold per muscle — the intensity used for the comparisons

Every muscle is recruited at a different intensity, so comparing all of them at one fixed mA mixes
muscles that are far above threshold with muscles that are still below it. Here each muscle gets
**its own motor threshold**: the lowest intensity at which its response appears — pulse-1
peak-to-peak above `MIN_SNR` × the pre-stimulus baseline, not artifact-rejected, and **still
passing at the next intensity** (`CONSECUTIVE = 2`), so a single noisy intensity cannot be
mistaken for a threshold.

Three steps: **detect** → **look at the recruitment curves and correct what is wrong** → **use it**.
The figure shows, per muscle, pulse-1 p2p against intensity: filled markers = passes the
criterion, hollow = does not, dashed line = the threshold that will be used.

In [ ]:

# ======== settings ========
CONSECUTIVE = 2            # intensities the response criterion must hold for
MT_SHOW     = ("burst", "before")   # the (mode, state) whose thresholds the figure below shows
# ==========================

MT_COL = {"cathodic": "#1f3b73", "anodic": "#e6550d"}

def mt_specs(mode, state):
    return [dict(label=pol, csv=D + FILE[(mode, pol, state)], colour=MT_COL[pol],
                 hatch=("" if pol == "cathodic" else "///")) for pol in ("cathodic", "anodic")]

MT = {}                    # MT[(mode, polarity, state)] = {muscle: mA}
for mode in ("burst", "arcex"):
    for state in ("before", "lidocaine"):
        got = motor_thresholds(mt_specs(mode, state), None, consecutive=CONSECUTIVE,
                               verbose=False, **KW)
        for pol in ("cathodic", "anodic"):
            MT[(mode, pol, state)] = got[pol]

print("motor threshold per muscle (mA) - first intensity with a response, held for "
      f"{CONSECUTIVE} steps\n")
names = sorted({m for d in MT.values() for m in d})
print(f"{'muscle':22s}" + "".join(f"{f'{m[:5]} {p[:4]} {s[:3]}':>16s}" for m, p, s in MT))
for n_ in names:
    print(f"{n_:22s}" + "".join(f"{str(MT[k].get(n_) or '-'):>16s}" for k in MT))


### Check the thresholds on the recruitment curves

Look for two mistakes: a threshold sitting on a **flat, noisy curve** (the criterion passed on
noise — typically at the lowest intensity tested), and a muscle with an obvious response whose
threshold was **not** found. Correct either in the next cell.

In [ ]:

mode_, state_ = MT_SHOW
fig_thresholds(mt_specs(mode_, state_), None, consecutive=CONSECUTIVE,
               MT={pol: MT[(mode_, pol, state_)] for pol in ("cathodic", "anodic")},
               title=f"{mode_} · {state_} lidocaine — motor threshold per muscle", **KW);


### Correct the thresholds by hand

`MT[polarity][muscle] = mA` sets one, `= None` drops that muscle from the comparison. Re-run the
figure above to check. Only muscles with a threshold in **both** polarities can be compared, and
those are listed by `muscles_with_threshold`.

In [ ]:

# --- manual corrections: MT[(mode, polarity, state)][muscle] = mA, or None to drop the muscle ---
# MT[("burst", "cathodic", "before")]["Triceps long (L)"] = None   # threshold on a flat, noisy curve
# MT[("burst", "anodic",   "before")]["Flex. digitorum (R)"] = 30
# ------------------------------------------------------------------------------------------------

def mt_common(mode, state):
    """Muscles with a threshold in BOTH polarities for that (mode, state) - the comparable set."""
    a, b = MT[(mode, "cathodic", state)], MT[(mode, "anodic", state)]
    return [m for m in sorted(a) if a.get(m) and b.get(m)]

for mode in ("burst", "arcex"):
    for state in ("before", "lidocaine"):
        ms = mt_common(mode, state)
        print(f"{mode:6s} {state:10s} {len(ms):2d} comparable muscles: " + ", ".join(ms))


### The comparison, each muscle at its own threshold

Same two measures as below — **2nd pulse vs 1st** and **mean of pulses 2–10 vs 1st** — but now
every muscle contributes at the intensity where *it* starts responding, instead of one mA for all.

In [ ]:
# the depression figure, with every muscle at its own motor threshold
MODE_DEP, STATE_DEP = MT_SHOW              # same (mode, state) as the threshold figure above
specs_dep = mt_specs(MODE_DEP, STATE_DEP)
for sp in specs_dep:
    sp["amp"] = MT[(MODE_DEP, sp["label"], STATE_DEP)]

table_mt = fig_depression(specs_dep, mt_common(MODE_DEP, STATE_DEP), metric="ratio",
                          title=f"{MODE_DEP} · {STATE_DEP} lidocaine — each muscle at its own motor threshold",
                          csv_out=None, save=None, **KW)

### The traces at those thresholds

`fig_train_modes` takes the same per-muscle intensities, so the paper figure can be drawn at
threshold too — pick 2–3 muscles.

In [ ]:
# the same trains as traces, for a few muscles
MT_FIG_MUSCLES = mt_common(MODE_DEP, STATE_DEP)[:3]
specs_tr = [dict(label=f"{sp['label']} · at MT", csv=sp["csv"], amp=sp["amp"], colour=sp["colour"],
                 linestyle=("-" if sp["label"] == "cathodic" else "--"), hatch=sp["hatch"])
            for sp in specs_dep]
fig_train_modes(specs_tr, MT_FIG_MUSCLES,
                title=f"{MODE_DEP} · {STATE_DEP} lidocaine · at each muscle's motor threshold", **KW);

## Figure 1 · 30 Hz vs ARC-EX — **anodic**, before and with lidocaine

Plain bars / solid lines = **30 Hz burst**, hatched / dashed = **ARC-EX**; gray = **before lidocaine**, orange = **with lidocaine**.

**Settings for this figure** — the mA of each compared condition, the muscle, the waterfall
muscles — are in `FIG_SETTINGS` in the config cell at the top.

In [ ]:
FIG1 = build(1)                 # settings: FIG_SETTINGS[1] in the config cell
# FIG1 = build(1, amps={...}, muscle="Biceps (R)")     # ...or override just here
check(FIG1)


### Figure 1a · all muscles

In [ ]:
show(FIG1, "Fig 1 · anodic — all muscles")


### Figure 1b · one muscle

In [ ]:
show(FIG1, f"Fig 1 · anodic — {FIG1['muscle']}", muscle=FIG1["muscle"])


### Figure 1c · how the peaks are detected (at the mA above)

In [ ]:
peaks(FIG1)                      # the figure's muscle; peaks(FIG1, muscle="Biceps (R)") for another
# peaks(FIG1, kw=dict(KW, resp_start_ms=11.0))    # try other detection settings


### Figure 1d · waterfalls — every intensity of each condition

One waterfall per condition, titled **mode · polarity · before / with lidocaine**, then the four
overlaid. All share one gain per muscle, so heights are comparable.

In [ ]:
wf(FIG1, "Fig 1 · anodic", muscles_=FIG_SETTINGS[1]["wf"])


## Figure 2 · 30 Hz vs ARC-EX — **cathodic**, before and with lidocaine

Plain / solid = **30 Hz burst**, hatched / dashed = **ARC-EX**; gray = **before lidocaine**, orange = **with lidocaine**.

**Settings for this figure** — the mA of each compared condition, the muscle, the waterfall
muscles — are in `FIG_SETTINGS` in the config cell at the top.

In [ ]:
FIG2 = build(2)                 # settings: FIG_SETTINGS[2] in the config cell
# FIG2 = build(2, amps={...}, muscle="Biceps (R)")     # ...or override just here
check(FIG2)


### Figure 2a · all muscles

In [ ]:
show(FIG2, "Fig 2 · cathodic — all muscles")


### Figure 2b · one muscle

In [ ]:
show(FIG2, f"Fig 2 · cathodic — {FIG2['muscle']}", muscle=FIG2["muscle"])


### Figure 2c · how the peaks are detected (at the mA above)

In [ ]:
peaks(FIG2)                      # the figure's muscle; peaks(FIG2, muscle="Biceps (R)") for another
# peaks(FIG2, kw=dict(KW, resp_start_ms=11.0))    # try other detection settings


### Figure 2d · waterfalls — every intensity of each condition

One waterfall per condition, titled **mode · polarity · before / with lidocaine**, then the four
overlaid. All share one gain per muscle, so heights are comparable.

In [ ]:
wf(FIG2, "Fig 2 · cathodic", muscles_=FIG_SETTINGS[2]["wf"])


## Figure 3 · ARC-EX — **cathodic vs anodic**, before and with lidocaine

Plain / solid = **cathodic**, hatched / dashed = **anodic**; gray = **before lidocaine**, orange = **with lidocaine**.

**Settings for this figure** — the mA of each compared condition, the muscle, the waterfall
muscles — are in `FIG_SETTINGS` in the config cell at the top.

In [ ]:
FIG3 = build(3)                 # settings: FIG_SETTINGS[3] in the config cell
# FIG3 = build(3, amps={...}, muscle="Biceps (R)")     # ...or override just here
check(FIG3)


### Figure 3a · all muscles

In [ ]:
show(FIG3, "Fig 3 · ARC-EX — all muscles")


### Figure 3b · one muscle

In [ ]:
show(FIG3, f"Fig 3 · ARC-EX — {FIG3['muscle']}", muscle=FIG3["muscle"])


### Figure 3c · how the peaks are detected (at the mA above)

In [ ]:
peaks(FIG3)                      # the figure's muscle; peaks(FIG3, muscle="Biceps (R)") for another
# peaks(FIG3, kw=dict(KW, resp_start_ms=11.0))    # try other detection settings


### Figure 3d · waterfalls — every intensity of each condition

One waterfall per condition, titled **mode · polarity · before / with lidocaine**, then the four
overlaid. All share one gain per muscle, so heights are comparable.

In [ ]:
wf(FIG3, "Fig 3 · ARC-EX", muscles_=FIG_SETTINGS[3]["wf"])


### Figure 3e · across intensities

In [ ]:
summary_curves(FIG3["files"], None, labels=FIG3["labels"], colours=FIG3["colours"],
               markers=["o", "o", "s", "s"], **KW);


## Figure 4 · 30 Hz burst — **cathodic vs anodic**, before and with lidocaine

Plain / solid = **cathodic**, hatched / dashed = **anodic**; gray = **before lidocaine**, orange = **with lidocaine**.

**Settings for this figure** — the mA of each compared condition, the muscle, the waterfall
muscles — are in `FIG_SETTINGS` in the config cell at the top.

In [ ]:
FIG4 = build(4)                 # settings: FIG_SETTINGS[4] in the config cell
# FIG4 = build(4, amps={...}, muscle="Biceps (R)")     # ...or override just here
check(FIG4)


### Figure 4a · all muscles

In [ ]:
show(FIG4, "Fig 4 · 30 Hz burst — all muscles")


### Figure 4b · one muscle

In [ ]:
show(FIG4, f"Fig 4 · 30 Hz burst — {FIG4['muscle']}", muscle=FIG4["muscle"])


### Figure 4c · how the peaks are detected (at the mA above)

In [ ]:
peaks(FIG4)                      # the figure's muscle; peaks(FIG4, muscle="Biceps (R)") for another
# peaks(FIG4, kw=dict(KW, resp_start_ms=11.0))    # try other detection settings


### Figure 4d · waterfalls — every intensity of each condition

One waterfall per condition, titled **mode · polarity · before / with lidocaine**, then the four
overlaid. All share one gain per muscle, so heights are comparable.

In [ ]:
wf(FIG4, "Fig 4 · 30 Hz burst", muscles_=FIG_SETTINGS[4]["wf"])


### Figure 4e · across intensities

In [ ]:
summary_curves(FIG4["files"], None, labels=FIG4["labels"], colours=FIG4["colours"],
               markers=["o", "o", "s", "s"], **KW);



## Paper-style figure — 30 Hz vs ARC-EX, before and with lidocaine

Four conditions on one figure: **black = 30 Hz burst, red = ARC-EX** (mode) · **solid, plain bars
= before lidocaine; dashed, hatched = with lidocaine**.

**Left** — the 10 pulses of each condition, one row per muscle, with a mV scale bar; the mA used
by each condition for *that* muscle is written in its colour at the top right of the panel.
**Right** — pulse 1 (outlined, = 100 %) vs the mean of pulses 2–10 (filled), each condition
relative to its own pulse 1; bar = mean over the muscles, whisker = SD, dots = the muscles, thin
grey lines join the same muscle across conditions.

`PAPER` below sets **the intensity per muscle and per protocol** — one entry per muscle, or
`"default"` for all of them. Set `SAVE` to write a 300 dpi PNG.


In [ ]:

# ======== paper figure settings ========
PAPER_POLARITY = "anodic"          # "anodic" or "cathodic"
PAPER_MUSCLES  = ["Flex. digitorum (R)", "Flex. carpi rad. (R)", "Ext. digitorum (L)"]

PAPER_MA = {   # intensity per protocol, per muscle ("default" covers the rest)
    "burst": {"Flex. digitorum (R)": 35, "Flex. carpi rad. (R)": 35, "Ext. digitorum (L)": 35,
              "default": 35},
    "arcex": {"Flex. digitorum (R)": 110, "Flex. carpi rad. (R)": 105, "Ext. digitorum (L)": 110,
              "default": 110},
}
SAVE = None                        # e.g. "figures/paper_burst_vs_arcex_anodic.png"
# =======================================

specs = [
    dict(label="30 Hz burst · before lido", csv=D + FILE[("burst", PAPER_POLARITY, "before")],
         amp=PAPER_MA["burst"], colour="black",   linestyle="-",  hatch=""),
    dict(label="30 Hz burst · with lido",   csv=D + FILE[("burst", PAPER_POLARITY, "lidocaine")],
         amp=PAPER_MA["burst"], colour="black",   linestyle="--", hatch="///"),
    dict(label="ARC-EX · before lido",      csv=D + FILE[("arcex", PAPER_POLARITY, "before")],
         amp=PAPER_MA["arcex"], colour="#d62728", linestyle="-",  hatch=""),
    dict(label="ARC-EX · with lido",        csv=D + FILE[("arcex", PAPER_POLARITY, "lidocaine")],
         amp=PAPER_MA["arcex"], colour="#d62728", linestyle="--", hatch="///"),
]
rest = fig_train_modes(specs, PAPER_MUSCLES, n_pulses=N_PULSES, resp_start_ms=RESP_START_MS,
                       guard_ms=GUARD_MS, title=f"{PAPER_POLARITY} - 30 Hz vs ARC-EX", save=SAVE)
for k, v in rest.items():
    print(f"{k:28s} mean of pulses 2-{N_PULSES} = {np.round(v, 0)} % of pulse 1  (per muscle)")


## Baseline cathodic vs anodic — depression along the train (all pre-lidocaine)

The two numbers asked for, per muscle: **2nd pulse vs 1st** and **mean of pulses 2–10 vs 1st**,
as % of the first pulse (100 % = no change, below = the response drops after the first pulse).
One figure per stimulation mode; only muscles that respond in **both** polarities are used, so the
two bars are paired — the grey lines join the same muscle. Everything here is **before lidocaine**.

`DEP_MA` sets the intensity of each mode, `metric="diff"` switches the y-axis to mV instead of %,
and `csv_out=` writes the per-muscle table for the paper.

In [ ]:
# ======== settings ========
DEP_MA      = {"burst": 35, "arcex": 110}     # intensity per mode (same for both polarities)
DEP_MUSCLES = None                            # None = all channels, or a list of labels
DEP_METRIC  = "ratio"                         # "ratio" = % of pulse 1 | "diff" = mV difference
DEP_SAVE    = False                           # True -> figures/ and results/ files
# ==========================

DEP_COL = {"cathodic": "#1f3b73", "anodic": "#e6550d"}
tables = {}
for mode, name in (("burst", "30 Hz burst"), ("arcex", "ARC-EX")):
    ma = DEP_MA[mode]
    specs = [dict(label="cathodic", csv=D + FILE[(mode, "cathodic", "before")], amp=ma,
                  colour=DEP_COL["cathodic"], hatch=""),
             dict(label="anodic",   csv=D + FILE[(mode, "anodic", "before")],   amp=ma,
                  colour=DEP_COL["anodic"],   hatch="///")]
    tables[mode] = fig_depression(
        specs, DEP_MUSCLES, n_pulses=N_PULSES, resp_start_ms=RESP_START_MS, guard_ms=GUARD_MS,
        min_snr=MIN_SNR, max_edge_frac=MAX_EDGE_FRAC, metric=DEP_METRIC,
        title=f"{name} · {ma} mA · pre-lidocaine — cathodic vs anodic",
        csv_out=(f"results/depression_{mode}_{ma}mA_pre.csv" if DEP_SAVE else None),
        save=(f"figures/depression_{mode}_{ma}mA_pre.png" if DEP_SAVE else None))
    print()


### The same numbers as a table

`depression_stats` returns one row per condition × muscle: pulse-1, pulse-2 and mean(2–10) in mV,
their differences from pulse 1, and the two percentages.

In [ ]:
import pandas as pd
df = pd.concat({m: pd.DataFrame(t) for m, t in tables.items()}, names=["mode"]).reset_index(level=0)
pd.set_option("display.width", 160, "display.max_columns", 20)
print(df.round(3).to_string(index=False))
